# Mathematical Derivations: Gradient Boosting Formulas

## Objective: Prove gain formula from first principles

---

## Part 1: Loss Function & Its Derivatives

### Binary Logistic Regression Loss

```
ℓ(y, ŷ) = -y·log(σ(ŷ)) - (1-y)·log(1-σ(ŷ))
```

where:
- `y ∈ {0, 1}` is the true label
- `ŷ ∈ ℝ` is the raw prediction (log-odds, **not** probability)
- `σ(ŷ) = 1 / (1 + e^(-ŷ))` is the sigmoid function

### Lemma 1: Derivative of Sigmoid

```
σ'(ŷ) = dσ/dŷ = σ(ŷ) · (1 - σ(ŷ))
```

**Proof**:
```
σ(ŷ) = (1 + e^(-ŷ))^(-1)

σ'(ŷ) = -1 · (1 + e^(-ŷ))^(-2) · (-e^(-ŷ))
      = e^(-ŷ) / (1 + e^(-ŷ))²
      = [1 / (1 + e^(-ŷ))] · [e^(-ŷ) / (1 + e^(-ŷ))]
      = σ(ŷ) · (1 - σ(ŷ))   ✓
```

### Theorem 1: Gradient of Logistic Loss

```
g = ∂ℓ/∂ŷ = σ(ŷ) - y
```

**Proof**:

Taking the derivative term-by-term:

```
∂ℓ/∂ŷ = ∂/∂ŷ [-y·log(σ(ŷ)) - (1-y)·log(1-σ(ŷ))]

     = -y · (1/σ(ŷ)) · σ'(ŷ) - (1-y) · (1/(1-σ(ŷ))) · (-σ'(ŷ))

     = -y · σ'(ŷ) / σ(ŷ) + (1-y) · σ'(ŷ) / (1-σ(ŷ))

Substituting σ'(ŷ) = σ(ŷ)(1 - σ(ŷ)):

     = -y · σ(ŷ)(1-σ(ŷ)) / σ(ŷ) + (1-y) · σ(ŷ)(1-σ(ŷ)) / (1-σ(ŷ))

     = -y(1 - σ(ŷ)) + (1-y)σ(ŷ)

     = -y + yσ(ŷ) + σ(ŷ) - yσ(ŷ)

     = σ(ŷ) - y   ✓
```

### Theorem 2: Hessian of Logistic Loss

```
h = ∂²ℓ/∂ŷ² = σ(ŷ) · (1 - σ(ŷ))
```

**Proof**:

```
h = ∂g/∂ŷ = ∂(σ(ŷ) - y) / ∂ŷ

  = ∂σ(ŷ)/∂ŷ - 0

  = σ'(ŷ)

  = σ(ŷ)(1 - σ(ŷ))   ✓
```

**Property**: Hessian is **always positive**, so loss is convex.

---

## Part 2: Second-Order Taylor Approximation

### Taylor Series Expansion

For a smooth function `L(ŷ)`, the Taylor expansion around `ŷ₀` is:

```
L(ŷ) ≈ L(ŷ₀) + g(ŷ₀)·(ŷ - ŷ₀) + (1/2)·h(ŷ₀)·(ŷ - ŷ₀)²
```

where:
- `g(ŷ₀) = ∂L/∂ŷ|_{ŷ₀}` (gradient at ŷ₀)
- `h(ŷ₀) = ∂²L/∂ŷ²|_{ŷ₀}` (Hessian at ŷ₀)

### Gradient Boosting Interpretation

In gradient boosting, we construct an ensemble:

```
ŷ^(t) = ŷ^(t-1) + η·f_t(x)
```

where:
- `ŷ^(t-1)` is the previous ensemble prediction
- `η` is the learning rate (shrinkage)
- `f_t(x)` is the new tree's prediction for sample x

**The tree's goal**: Minimize the combined loss

```
L_total = Σ_i ℓ(y_i, ŷ^(t-1)_i + η·f_t(x_i))
```

Expand each term using Taylor (where `Δ = η·f_t(x_i)`):

```
ℓ(y_i, ŷ^(t-1)_i + Δ) ≈ ℓ(y_i, ŷ^(t-1)_i) + g_i · Δ + (1/2)·h_i · Δ²
```

Substituting `Δ = η·f_t(x_i)`:

```
ℓ_i ≈ const + g_i · η·f_t(x_i) + (1/2)·h_i · η²·f_t(x_i)²
```

**Dropping constants and absorbing η**:

```
ℓ_i ≈ g_i · f_t(x_i) + (1/2)·h_i · f_t(x_i)²
```

This is the **residual-fitting interpretation**: the tree fits gradients `g_i`, weighted by Hessians `h_i`.

---

## Part 3: Gain Formula Derivation

### Setting: One Split

Consider a set `S` of n samples at a node. We split using a feature/threshold:
- Left child: subset `L` of n_L samples
- Right child: subset `R` of n_R samples
- `S = L ∪ R`, `|S| = n = n_L + n_R`

### Objective Before Split

Optimize constant predictions `w_L`, `w_R` for each child:

```
Loss_after = (1/2)·Σ_{i ∈ L} h_i·w_L² + (1/2)·Σ_{i ∈ L} h_i·g_i·w_L
           + (1/2)·Σ_{i ∈ R} h_i·w_R² + (1/2)·Σ_{i ∈ R} h_i·g_i·w_R
           + λ·(|w_L| + |w_R|)   [L1 regularization, often omitted for simplicity]
```

For L2 regularization only:

```
Loss_after = (1/2)·Σ_{i ∈ L} h_i·w_L² + (1/2)·Σ_{i ∈ L} h_i·g_i·w_L
           + (1/2)·Σ_{i ∈ R} h_i·w_R² + (1/2)·Σ_{i ∈ R} h_i·g_i·w_R
           + (λ/2)·(w_L² + w_R²)
```

### Optimal Leaf Weights

Minimize w.r.t. w_L:

```
∂Loss_after/∂w_L = Σ_{i ∈ L} h_i·w_L + Σ_{i ∈ L} h_i·g_i + λ·w_L = 0

⟹ w_L·(H_L + λ) = -G_L

⟹ w_L* = -G_L / (H_L + λ)
```

Similarly:

```
w_R* = -G_R / (H_R + λ)
```

where `G_L = Σ_{i∈L} g_i`, `H_L = Σ_{i∈L} h_i`, etc.

### Optimal Loss Value After Split

Substituting `w_L*` and `w_R*` back:

```
Loss_after = -G_L² / (2(H_L + λ)) - G_R² / (2(H_R + λ))
```

### Objective Before Split (No Split Case)

If we don't split and keep all samples in one leaf:

```
Loss_before = -G² / (2(H + λ))
```

where `G = G_L + G_R`, `H = H_L + H_R`.

### Gain Definition

The **gain** is the reduction in loss:

```
Gain = Loss_before - Loss_after

     = -G² / (2(H + λ)) + G_L² / (2(H_L + λ)) + G_R² / (2(H_R + λ))

     = (1/2) · [G_L² / (H_L + λ) + G_R² / (H_R + λ) - G² / (H + λ)]
```

**Simplifying (absorb the 1/2 into choice of hyperparameter units)**:

```
Gain = G_L² / (H_L + λ) + G_R² / (H_R + λ) - G² / (H + λ)   ✓
```

This is the **XGBoost gain formula**.

---

## Part 4: Intuition Behind Each Term

### Term 1: `G_L² / (H_L + λ)`

Represents the "purity" of the left child:
- High when `|G_L|` is large (homogeneous label distribution)
- Normalized by `H_L` (dividing by sample confidence)
- Regularization λ prevents overfitting (penalizes too-confident leaves)

### Term 2: `G_R² / (H_R + λ)`

Same interpretation for the right child.

### Term 3: `G² / (H + λ)`

The baseline cost of not splitting (keeping all samples together).

### Net Effect

A split is good if `Term1 + Term2 > Term3`:
- The split creates two homogeneous children (high left/right terms)
- Better than keeping all heterogeneous samples together (high baseline term)

---

## Part 5: Special Cases

### Case 1: Unregularized (λ = 0)

```
Gain = G_L²/H_L + G_R²/H_R - G²/H

     = (G_L²·H_R + G_R²·H_L - G²·H_R·H_L/H) / (H_L·H_R)
```

If all samples have equal Hessians (h_i = h), then `H_L = h·n_L`, etc.:

```
Gain ≈ (G_L² + G_R²) / h - G² / h

     ∝ Σ g_i² in each region  [variance of residuals]
```

This shows gain is related to variance reduction—a key property of tree splitting.

### Case 2: Pure Classification (σ constant)

If all sample confidences are equal (uniform σ, uniform Hessian h):

```
H_L = h·n_L, H_R = h·n_R, H = h·n

Gain = G_L²/(h·n_L) + G_R²/(h·n_R) - G²/(h·n)

     = (1/h) · [G_L²/n_L + G_R²/n_R - G²/n]
```

The gain depends on **balancing group means** (standard classification criterion).

### Case 3: Perfectly Separated Split

If the split completely separates classes (all y=0 on left, all y=1 on right):

```
G_L = 0.5·n_L (all y=0, σ=0.5)
G_R = -0.5·n_R (all y=1, σ=0.5)
G = 0.5·n_L - 0.5·n_R = 0 (if balanced)

Gain = 0 + 0.5²·n_R/(h·n_R + λ) - 0 = 0.25·n_R/(h·n_R + λ) >> 0
```

Perfect separation yields **maximum gain** ✓

---

## Part 6: Connections to Other Frameworks

### Decision Trees (Gini Impurity)

Standard decision trees use Gini or entropy-based splits:

```
Gini_impurity = 1 - Σ p_c²   [for each class c]
```

**Relationship**: XGBoost gain generalizes Gini to gradient boosting context:
- Gini measures *class balance*
- XGBoost gain measures *gradient alignment* (more flexible)

### Information Theory (Cross-Entropy)

Information gain (ID3, C4.5 algorithms):

```
IG = H(S) - Σ_{v} (|S_v|/|S|) · H(S_v)
```

**Relationship**: Information gain uses class entropy; XGBoost gain uses gradient variance. Both are information-theoretic but in different spaces.

---

## Part 7: Regularization Terms Deep Dive

### L2 Regularization (λ in denominator)

```
w* = -G / (H + λ)
```

**Effect**: Adding λ to the denominator **shrinks** leaf weights toward zero.

**Example**:
- Without λ: `w = -10 / 5 = -2.0`
- With λ=5: `w = -10 / 10 = -1.0`

Prevents **over-confident trees** on small, noisy data.

### L1 Regularization (α)

```
w* = sign(G) · max(0, |G| / (H + λ) - α)
```

**Effect**: Sets small weights exactly to zero (feature selection).

Used less frequently in XGBoost; L2 is default.

### Gamma (Minimum Split Gain)

```
Accept split only if Gain ≥ γ
```

**Effect**: Prevents splits that don't significantly improve loss.

Threshold γ controls tree depth indirectly.

---

## Part 8: Multi-Class Extension

### One-vs-Rest Setup

For K-class classification, train K binary classifiers:
- Class k vs. rest

Each gets its own gradients:

```
g_i^(k) = P(y_i = k | x_i) - δ(y_i = k)
```

where δ(·) is indicator function.

The gain formula remains identical; only `g_i` and `h_i` change.

### Softmax Regression (Multinomial)

Loss: `ℓ = -log(softmax_c(ŷ))`

Gradients per class:

```
g_i^(c) = softmax_c(ŷ) - δ(y_i = c)
h_i^(c) = softmax_c(ŷ) · (1 - softmax_c(ŷ))   [diagonal]
```

Cross-class Hessian terms exist but typically ignored in tree splitting (approximate).

---

## Part 9: Connection to Newton's Method

### Newton's Method for Optimization

To minimize loss `L(w)`:

```
w_{t+1} = w_t - H^{-1} · ∇L
```

where `H` is the Hessian matrix.

### XGBoost as Newton Boosting

The leaf weight update:

```
w* = -G / H
```

is a **single Newton step**, assuming constant Hessian within the leaf.

**Interpretation**: Each tree is a Newton step, but performed greedily (feature-by-feature splits) rather than full matrix inversion.

This is why XGBoost often converges faster than gradient descent (which uses only first-order info).

---

## Summary Table: Formula Recap

| Concept | Formula | Interpretation |
|---------|---------|-----------------|
| **Gradient** | `g = σ - y` | Direction & magnitude of error |
| **Hessian** | `h = σ(1-σ)` | Curvature (confidence uncertainty) |
| **Leaf Weight** | `w = -G / (H + λ)` | Optimal prediction (Newton step) |
| **Gain** | `G_L²/(H_L+λ) + G_R²/(H_R+λ) - G²/(H+λ)` | Loss reduction from split |
| **Gain > γ** | Split accepted | Minimum split quality threshold |